In [6]:
!python -m deeppavlov install insults_kaggle_bert


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Ignoring transformers: markers 'python_version < "3.8"' don't match your environment

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
!python -m deeppavlov install rusentiment_bert


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Ignoring transformers: markers 'python_version < "3.8"' don't match your environment

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
!pip install pymorphy2 pymorphy2-dicts-ru

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 795.4 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 6.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=e163163669b004ba4abd660f90f6ff04776e9dcc7c55827ca686c3610d67b9ba
  Stored in directory: /root/.cache/pip/wheels/70/4a/46/1309fc853b8d395e60bafaf1b6df7845bdd82c95fd59dd8d2b
Successfully built docopt

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
import pymorphy2
from deeppavlov import build_model

# 1. Инициализация лемматизатора для русского языка
morph = pymorphy2.MorphAnalyzer()

def lemmatize_ru(text):
    if not isinstance(text, str): return ""
    return " ".join([morph.parse(word)[0].normal_form for word in text.split()])

# 2. Загрузка моделей DeepPavlov
model_en = build_model("insults_kaggle_bert", download=True)
model_ru = build_model("rusentiment_bert", download=True)

print("Модели загружены и готовы к работе!")

2026-03-11 12:41:37.518 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/datasets/insults_data.tar.gz download because of matching hashes
2026-03-11 12:41:48.56 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/deeppavlov_data/classifiers/insults_kaggle_torch_bert_v5.tar.gz download because of matching hashes
/usr/local/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initi

Модели загружены и готовы к работе!


In [2]:
import pandas as pd
import pymorphy2
from deeppavlov import build_model

# 1. Инициализация инструментов
morph = pymorphy2.MorphAnalyzer()

def lemmatize_ru(text):
    if not isinstance(text, str): return ""
    return " ".join([morph.parse(word)[0].normal_form for word in text.split()])

# Загружаем модели
model_en = build_model("insults_kaggle_bert", download=True)
model_ru = build_model("rusentiment_bert", download=True)

#ЧАСТЬ 1: Английский язык

# Берем первые 500 строк, и указываем, что колонки могут быть "грязными"
df_en = pd.read_csv('toxic_comments.csv', nrows=500, on_bad_lines='skip', engine='python')

# Уберем пустые строки, чтобы модель не ругалась
df_en = df_en.dropna(subset=['text'])

# Берем 10 строк
texts_en = df_en['text'].head(10).astype(str).tolist()
df_en_sample = df_en.head(10).copy()
df_en_sample['predictions'] = model_en(texts_en)

# ЧАСТЬ 2: Русский язык

# Читаем русский датасет
df_ru = pd.read_csv('rusentitweet_full.csv', nrows=500, on_bad_lines='skip', engine='python')
df_ru = df_ru.dropna(subset=['text'])

# Берем 10 строками
df_ru_sample = df_ru.head(10).copy()

# Лемматизация
df_ru_sample['lemmatized_text'] = df_ru_sample['text'].apply(lemmatize_ru)

# Тональность
df_ru_sample['sentiment'] = model_ru(df_ru_sample['lemmatized_text'].tolist())

#ВЫВОД РЕЗУЛЬТАТОВ
print("\n--- Результаты Английский ---")
display(df_en_sample[['text', 'toxic', 'predictions']])

print("\n--- Результаты Русский ---")
display(df_ru_sample[['text', 'lemmatized_text', 'sentiment']])

2026-03-11 12:46:44.577 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/datasets/insults_data.tar.gz download because of matching hashes
2026-03-11 12:46:58.849 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/deeppavlov_data/classifiers/insults_kaggle_torch_bert_v5.tar.gz download because of matching hashes
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForP


--- Результаты Английский ---


,text,toxic,predictions
0,Explanation\nWhy the edits made under my usern...,0,Not Insult
1,D'aww! He matches this background colour I'm s...,0,Not Insult
2,"Hey man, I'm really not trying to edit war. It...",0,Not Insult
3,"""\nMore\nI can't make any real suggestions on ...",0,Not Insult
4,"You, sir, are my hero. Any chance you remember...",0,Not Insult
5,"""\n\nCongratulations from me as well, use the ...",0,Not Insult
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,Insult
7,Your vandalism to the Matt Shirvington article...,0,Not Insult
8,Sorry if the word 'nonsense' was offensive to ...,0,Not Insult
9,alignment on this subject and which are contra...,0,Not Insult



--- Результаты Русский ---


,text,lemmatized_text,sentiment
0,@varlamov @McFaul На,@varlamov @mcfaul на,neutral
1,велл они всё равно что мусор так что ничего с...,велла они всё равно что мусор так что ничего с...,negative
2,"""трезвая жизнь какая-то такая стрёмная""\r\n(с)...","""трезвый жизнь какой-то такой стрёмная"" (с) ар...",neutral
3,Ой какие неожиданные результаты 🤭 https://t.co...,ой какой неожиданный результат 🤭 https://t.co/...,neutral
4,@Shvonder_chief @dimsmirnov175 На заборе тоже ...,@shvonder_chief @dimsmirnov175 на забор тоже н...,neutral
5,@idkwhht мы тоже мебельная компания уджина😳😳😳,@idkwhht мы тоже мебельный компания уджина😳😳😳,positive
6,Счастья здоровья 10 классникам https://t.co/M9...,счастие здоровье 10 классник https://t.co/m9vu...,neutral
7,@dntbliev НЕ ПАЛИ.,@dntbliev не пали.,negative
8,@BTS_twt ты такой красивый 😭😭😭🥺💓,@bts_twt ты такой красивый 😭😭😭🥺💓,positive
9,"@Ladyzchensk Цыган , хуле ...","@ladyzchensk цыган , хула ...",negative
